# Лабораторная работа: Применение дерева решений для задач классификации
## Датасет: Seeds (UCI Machine Learning Repository)

### Пункт 4. Обучение модели

### 4.1. Загрузка и подготовка данных

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

# Загрузка датасета
columns_names = ['area', 'perimeter', 'compactness', 'kernel_length', 
                 'kernel_width', 'asymmetry_coeff', 'groove_length', 'target']

seed = 4403

df = pd.read_csv('Seed/Seed_Data.csv', header=0, names=columns_names)
df.head()

In [ ]:
# Проверка на пропущенные значения
print("Проверка на пропущенные значения:")
print(df.isnull().sum())

In [ ]:
# Описательная статистика
print("Описательная статистика:")
print(df.describe())

In [ ]:
# Сбалансированность классов
print("Распределение классов:")
print(df['target'].value_counts())

### 4.2. Разделение на обучающую и тестовую выборки

In [ ]:
# Разделение на признаки и целевую переменную
X = df.drop('target', axis=1)
y = df['target']

# Разделение на обучающую и тестовую выборки (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed
)

print(f"Размер обучающей выборки: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Размер тестовой выборки: X_test={X_test.shape}, y_test={y_test.shape}")

### 4.3. Построение дерева решений

In [ ]:
# Обучение базовой модели (без ограничения глубины)
dt_base = DecisionTreeClassifier(criterion='gini', random_state=seed)
dt_base.fit(X_train, y_train)
y_pred_base = dt_base.predict(X_test)

accuracy_base = accuracy_score(y_test, y_pred_base)
print(f"Точность базовой модели (без ограничения глубины): {accuracy_base:.4f}")

In [ ]:
# Эксперименты с ограничением глубины дерева
print("Сравнение точности при разной глубине дерева:")
print(f"{'Глубина':<10} {'Точность':<10}")
print("-" * 20)

results = []
best = {
    'depth': 0,
    'accuracy': 0
}
for depth in [2, 3, 4, 5]:
    dt = DecisionTreeClassifier(criterion='gini', max_depth=depth, random_state=seed)
    dt.fit(X_train, y_train)
    y_pred = dt.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append((depth, acc))
    
    if (acc > best['accuracy']):
        best['accuracy'] = acc
        best['depth'] = depth
    
    print(f"{depth:<10} {acc:.4f}")

In [ ]:
# Выбираем модель с наибольшей точностью
dt_best = DecisionTreeClassifier(criterion='gini', max_depth=best['depth'], random_state=seed)
dt_best.fit(X_train, y_train)
y_pred_best = dt_best.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)

print(f"Выбрана модель с max_depth={best['depth']}, точность: {accuracy_best:.4f}")

### 4.4. Визуализация дерева решений

In [ ]:
# Названия признаков на русском языке
feature_names_ru = [
    'Площадь',
    'Периметр',
    'Компактность',
    'Длина зерна',
    'Ширина зерна',
    'Коэф. асимметрии',
    'Длина бороздки'
]

class_names = ['Kama', 'Rosa', 'Canadian']

# Визуализация дерева
plt.figure(figsize=(25, 15))
plot_tree(dt_best, 
          feature_names=feature_names_ru,
          class_names=class_names,
          filled=True,
          rounded=True,
          fontsize=10)
plt.savefig('decision_tree.png', dpi=150, bbox_inches='tight')
plt.close()

print("Дерево сохранено в decision_tree.png")

# Отображение дерева в ноутбуке
from IPython.display import Image
Image(filename='decision_tree.png')

### 4.5. Оценка качества модели

In [ ]:
# Общая точность
print(f"Общая точность классификации: {accuracy_best:.4f}")

In [ ]:
# Матрица ошибок
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Матрица ошибок')
plt.show()

In [ ]:
# Подробный отчёт по каждому классу
print("Подробный отчёт по каждому классу:")
print(classification_report(y_test, y_pred_best, target_names=class_names))